In [6]:
import ase
import numpy as np
import pyscf
import time
import os
from pyscf import gto, dft, df, lib
from pyscf.scf import hf
import equiv_dens.utils.base as utils
import scipy
hf.MUTE_CHKFILE = True
%load_ext autoreload
%autoreload 2
%cd /home/mihail/Documents/workspace/equiv_dens/

/home/mihail/Documents/workspace/equiv_dens


In [15]:
# load data
data = np.load('md_logs/2022-12-23_pJuMyc8e/simulation_-ethanethiol_cluster_all_001_compressed_0.npy', allow_pickle=True).item()
data['positions'] = data['positions'][-100:]
print(data['positions'].shape)

(100, 9, 3)


In [ ]:
atom_pos = data['positions']
atom_types = data['atom_numbers'].squeeze()
save_path = 'datasets/ethanethiol_md_traj_dft_augccpvdz_df_augccpvqzjkfit.npy'
npy_path = 'datasets/ethanethiol_md_traj_dft_augccpvdz.npy'
if os.path.exists(save_path):
    results = list(np.load(save_path, allow_pickle=True))
else:
    results = []
print('results len', len(results))
basis = 'augccpvdz'
auxbasis = 'augccpvqzjkfit'
for i in range(len(results), len(atom_pos)):
    print('calc', i)
    start = time.time()
    pos = atom_pos[i]
    atom = []
    for j in range(len(atom_types)):
        atom.append((atom_types[j], pos[j, :]))
    print(atom)
    mol = gto.M(atom=atom, basis='augccpvdz')
    # print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile = False
    mf.xc = 'pbe'
    # mf.max_cycle = 1000
    mf.kernel()
    g = mf.nuc_grad_method()
    gradients = g.grad()
    # print(mfs[i].mo_coeff)
    res = []
    res.append(mol.pack())
    calc_dict = {}
    print('mo occ', mf.mo_occ)
    calc_dict['mo_coeff'] = mf.mo_coeff
    calc_dict['mo_occ'] = mf.mo_occ
    calc_dict['energy'] = mf.e_tot
    calc_dict['forces'] = -gradients/ase.units.Bohr

    dm1 = mf.make_rdm1(mf.mo_coeff, mf.mo_occ)
    auxmol = df.addons.make_auxmol(mol, auxbasis)

    ints_3c2e = df.incore.aux_e2(mol, auxmol, intor='int3c2e')
    ints_2c2e = auxmol.intor('int2c2e')
    print('ints3c2e shape', ints_3c2e.shape)
    print('ints2c2e shape', ints_2c2e.shape)

    nao = mol.nao
    naux = auxmol.nao
    df_coef = scipy.linalg.solve(ints_2c2e, ints_3c2e.reshape(nao*nao, naux).T)
    df_coef = df_coef.reshape(naux, nao, nao)
    if dm1.ndim > 2:
        df_basis = []
        for j in range(dm1.shape[0]):
            df_basis.append(lib.einsum('Pij,ij->P', df_coef, dm1[j]))
        df_basis = np.stack(df_basis, axis=0)
        print(df_basis.shape)

    else:
        df_basis = lib.einsum('Pij,ij->P', df_coef, dm1)

    calc_dict['df_coeff'] = df_basis
    calc_dict['auxbasis'] = auxbasis
    res.append(calc_dict)
    results.append(res)
    if (i+1) % 1000 == 0:
        print('i=', i, 'saving file')
        # if (i+1) == 4000:
        #     break
        np.save(save_path, results, allow_pickle=True)
np.save(save_path, results, allow_pickle=True)
npy_data = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=False)
np.save(npy_path, npy_data, allow_pickle=True)

results len 0
calc 0
[(1, array([-61.297337, 160.98715 ,  40.097767], dtype=float32)), (1, array([-61.215813, 161.67055 ,  41.683994], dtype=float32)), (1, array([-60.604805, 163.97736 ,  40.688267], dtype=float32)), (1, array([-59.5714  , 162.76228 ,  40.158867], dtype=float32)), (1, array([-60.85352 , 163.38254 ,  39.047897], dtype=float32)), (1, array([-63.33516, 162.3852 ,  39.12311], dtype=float32)), (6, array([-61.473633, 161.91165 ,  40.61054 ], dtype=float32)), (6, array([-60.5966  , 163.0709  ,  40.097008], dtype=float32)), (16, array([-63.299175, 162.31947 ,  40.51998 ], dtype=float32))]
converged SCF energy = -477.728019790096
--------------- RKS gradients ---------------
         x                y                z
0 H    -0.0031745574     0.0163393862     0.0163814358
1 H     0.0087907486    -0.0074315483     0.0099680309
2 H     0.0112636372    -0.0105505816    -0.0128407135
3 H    -0.0246153706     0.0095842426     0.0064854239
4 H    -0.0037580944     0.0072550818    -0

calc 5
[(1, array([-61.302258, 160.99637 ,  40.018192], dtype=float32)), (1, array([-61.222176, 161.68146 ,  41.663292], dtype=float32)), (1, array([-60.65864 , 163.96758 ,  40.670288], dtype=float32)), (1, array([-59.560417, 162.73323 ,  40.116158], dtype=float32)), (1, array([-60.84134 , 163.33022 ,  39.094566], dtype=float32)), (1, array([-63.336567, 162.47073 ,  39.201817], dtype=float32)), (6, array([-61.47848 , 161.9144  ,  40.597572], dtype=float32)), (6, array([-60.59697, 163.06784,  40.09422], dtype=float32)), (16, array([-63.299503, 162.32892 ,  40.52971 ], dtype=float32))]
converged SCF energy = -477.730050076964
--------------- RKS gradients ---------------
         x                y                z
0 H     0.0002331233     0.0021009006     0.0022633505
1 H     0.0072337866    -0.0019455678     0.0073192990
2 H     0.0054418618    -0.0168536839    -0.0172171204
3 H    -0.0127088818     0.0025897454     0.0011321118
4 H     0.0065987363    -0.0054095550     0.0300333485
5 

calc 10
[(1, array([-61.307816, 160.99896 ,  39.932354], dtype=float32)), (1, array([-61.249462, 161.69746 ,  41.624393], dtype=float32)), (1, array([-60.727848, 163.99951 ,  40.6977  ], dtype=float32)), (1, array([-59.514183, 162.6976  ,  40.070282], dtype=float32)), (1, array([-60.845383, 163.29071 ,  39.06504 ], dtype=float32)), (1, array([-63.34018, 162.56161,  39.23863], dtype=float32)), (6, array([-61.483345, 161.92003 ,  40.586185], dtype=float32)), (6, array([-60.59735 , 163.05885 ,  40.093925], dtype=float32)), (16, array([-63.299194, 162.33786 ,  40.54106 ], dtype=float32))]
converged SCF energy = -477.72967058727
--------------- RKS gradients ---------------
         x                y                z
0 H     0.0038377501    -0.0143574989    -0.0156745208
1 H     0.0009198269     0.0066038969    -0.0104263695
2 H    -0.0048738813     0.0118134632     0.0047632104
3 H     0.0160647366    -0.0101141711    -0.0050910047
4 H     0.0035606754    -0.0054554317     0.0145627548
5 